# 01 — F1 Data Collection (2022–2025) — CORRECTED

Collects lap-by-lap race data from FastF1 3.8.3 for all seasons 2022–2025.
Uses verified column names directly from FastF1 API documentation.

**Output:** `data/raw/raw_f1_data_2022_2025.csv`

## 0. Imports & Configuration

In [44]:
import fastf1
import pandas as pd
import numpy as np
import os
import time
import logging
from pathlib import Path
from tqdm.notebook import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent  
CACHE_DIR    = PROJECT_ROOT / 'data' / 'fastf1_cache'
RAW_DIR      = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_FILE  = RAW_DIR / 'raw_f1_data_2022_2025.csv'
PROGRESS_FILE = RAW_DIR / '_collection_progress.csv'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── FastF1 cache ───────────────────────────────────────────────────────────
fastf1.Cache.enable_cache(str(CACHE_DIR))
logging.basicConfig(level=logging.WARNING)

# ── Seasons & config ───────────────────────────────────────────────────────
SEASONS = [2025]  # Start with just 2022
DELAY_BETWEEN_SESSIONS = 10  # seconds

print(f'FastF1 version: {fastf1.__version__}')
print(f'Output file   : {OUTPUT_FILE}')

FastF1 version: 3.8.3
Output file   : /Users/sara/Desktop/F1_May2026/data/raw/raw_f1_data_2022_2025.csv


## 1. Build Race Schedule

In [45]:
def get_race_schedule(seasons):
    """Return list of (year, round, event_name, country, location, date) for all races."""
    schedule = []
    for year in seasons:
        try:
            events = fastf1.get_event_schedule(year, include_testing=False)
            races = events[events['RoundNumber'] > 0]
            for _, row in races.iterrows():
                schedule.append({
                    'season': year,
                    'round': int(row['RoundNumber']),
                    'event_name': row['EventName'],
                    'country': row.get('Country', ''),
                    'location': row.get('Location', ''),
                    'date': str(row.get('EventDate', ''))[:10],
                })
            print(f'{year}: {len(races)} races found')
        except Exception as e:
            print(f'{year}: ERROR fetching schedule — {e}')
    return pd.DataFrame(schedule)

schedule_df = get_race_schedule(SEASONS)
print(f'\nTotal races: {len(schedule_df)}')
schedule_df.head()

2025: 24 races found

Total races: 24


,season,round,event_name,country,location,date
0,2025,1,Australian Grand Prix,Australia,Melbourne,2025-03-16
1,2025,2,Chinese Grand Prix,China,Shanghai,2025-03-23
2,2025,3,Japanese Grand Prix,Japan,Suzuka,2025-04-06
3,2025,4,Bahrain Grand Prix,Bahrain,Sakhir,2025-04-13
4,2025,5,Saudi Arabian Grand Prix,Saudi Arabia,Jeddah,2025-04-20


## 2. Define Data Extraction Function

Uses verified FastF1 3.8.3 column names from official documentation.

In [46]:
STREET_CIRCUITS = {
    'Monaco Grand Prix',
    'Las Vegas Grand Prix',
    'Abu Dhabi Grand Prix',
    'Australian Grand Prix',
    'Azerbaijan Grand Prix',
    'Miami Grand Prix',
    'Singapore Grand Prix',
    'Saudi Arabian Grand Prix',
}

TRACK_STATUS_MAP = {
    '1': 'Clear', '2': 'Yellow', '4': 'SafetyCar', 
    '5': 'RedFlag', '6': 'VSC', '7': 'VSCEnding',
}

def extract_race_data(year, round_number, event_name, country, location, date):
    """
    Load a single race and return flat DataFrame.
    Uses verified FastF1 column names from API docs.
    """
    try:
        session = fastf1.get_session(year, round_number, 'R')
        session.load(laps=True, telemetry=False, weather=True, messages=False)
    except Exception as e:
        print(f'  LOAD ERROR {year} R{round_number}: {e}')
        return None

    try:
        laps = session.laps.copy()
        if laps.empty:
            return None

        # ── Race-level metadata ────────────────────────────────────────────
        laps['season'] = year
        laps['round'] = round_number
        laps['event_name'] = event_name
        laps['country'] = country
        laps['location'] = location
        laps['date'] = date
        laps['is_street_circuit'] = event_name in STREET_CIRCUITS
        laps['total_race_laps'] = laps['LapNumber'].max()

        # ── Merge race results ─────────────────────────────────────────────
        # session.results columns: DriverNumber, Abbreviation, TeamName, GridPosition, Position, Status, Points, Laps
        results = session.results[['DriverNumber', 'Abbreviation', 'TeamName', 'GridPosition', 'Position', 'Status', 'Points', 'Laps']].copy()
        results.columns = ['DriverNumber', 'driver', 'team', 'grid_position', 'final_position', 'status', 'points_scored', 'laps_completed']
        laps = laps.merge(results, on='DriverNumber', how='left')

        # ── DNF encoding ───────────────────────────────────────────────────
        def is_dnf(status):
            if pd.isna(status):
                return True
            s = str(status).strip()
            return not (s == 'Finished' or (s.startswith('+') and 'Lap' in s))

        laps['dnf'] = laps['status'].apply(is_dnf)
        laps['dnf_reason'] = laps['status'].where(laps['dnf'], other=pd.NA)
        dnf_laps = laps[laps['dnf']].groupby('Driver')['LapNumber'].max().rename('dnf_lap')
        laps = laps.merge(dnf_laps, on='Driver', how='left')
        laps.loc[~laps['dnf'], 'dnf_lap'] = pd.NA

        # ── Track status flags (native FastF1) ──────────────────────────────
        ts = laps['TrackStatus'].astype(str)
        laps['track_status_code'] = ts
        laps['track_status'] = ts.map(TRACK_STATUS_MAP).fillna('Unknown')
        laps['is_yellow_flag'] = ts == '2'
        laps['is_safety_car'] = ts == '4'
        laps['is_vsc'] = ts.isin(['6', '7'])
        laps['is_red_flag'] = ts == '5'

        # ── Pit stop (native FastF1 columns) ───────────────────────────────
        laps['is_pit_lap'] = laps['PitInTime'].notna()
        laps['pit_in_time'] = laps['PitInTime'].apply(
            lambda x: x.total_seconds() if pd.notna(x) else np.nan
        )
        laps['pit_out_time'] = laps['PitOutTime'].apply(
            lambda x: x.total_seconds() if pd.notna(x) else np.nan
        )

        # ── Weather (join from session.weather_data by session time) ──────────
        try:
            weather = session.weather_data
            if weather is not None and not weather.empty:
                # Merge nearest weather reading to each lap's session time
                laps_time = laps[['Time']].copy().reset_index(drop=False)
                merged = pd.merge_asof(
                    laps_time.sort_values('Time'),
                    weather[['Time', 'AirTemp', 'TrackTemp', 'Humidity', 'Rainfall']],
                    on='Time',
                    direction='nearest'
                )
                # Restore original order
                merged = merged.sort_values('index').set_index('index')
                laps['air_temp']   = merged['AirTemp']
                laps['track_temp'] = merged['TrackTemp']
                laps['humidity']   = merged['Humidity']
                laps['rainfall']   = merged['Rainfall']
            else:
                raise ValueError('No weather data')
        except Exception:
            laps['air_temp']   = np.nan
            laps['track_temp'] = np.nan
            laps['humidity']   = np.nan
            laps['rainfall']   = False

        laps['is_wet_race'] = laps['rainfall'].fillna(False).astype(bool).any()

        # ── Rename remaining native FastF1 fields ──────────────────────────
        laps = laps.rename(columns={
            'DriverNumber': 'driver_number',
            'Driver': 'driver_abbrev',  # 3-letter code
            'LapNumber': 'lap_number',
            'Position': 'position',
            'LapTime': 'lap_time_raw',
            'Sector1Time': 'sector1_raw',
            'Sector2Time': 'sector2_raw',
            'Sector3Time': 'sector3_raw',
            'Compound': 'tire_compound',
            'TyreLife': 'tire_age',
            'FreshTyre': 'is_fresh_tire',
            'Stint': 'stint',
            'Team': 'team_laps',  # from laps, different from results.TeamName
            'IsAccurate': 'is_accurate',
        })

        # Convert timedeltas to seconds
        for raw_col, sec_col in [
            ('lap_time_raw', 'lap_time_seconds'),
            ('sector1_raw', 'sector1_seconds'),
            ('sector2_raw', 'sector2_seconds'),
            ('sector3_raw', 'sector3_seconds'),
        ]:
            if raw_col in laps.columns:
                laps[sec_col] = laps[raw_col].apply(
                    lambda x: x.total_seconds() if pd.notna(x) else np.nan
                )
            else:
                laps[sec_col] = np.nan

        # ── Select final columns ───────────────────────────────────────────
        final_cols = [
            'season', 'round', 'event_name', 'country', 'location', 'date',
            'is_street_circuit', 'total_race_laps',
            'driver', 'driver_number', 'driver_abbrev', 'team',
            'grid_position', 'final_position', 'points_scored', 'laps_completed',
            'status', 'dnf', 'dnf_reason', 'dnf_lap',
            'lap_number', 'position',
            'lap_time_seconds', 'sector1_seconds', 'sector2_seconds', 'sector3_seconds',
            'tire_compound', 'tire_age', 'is_fresh_tire', 'stint',
            'is_pit_lap', 'pit_in_time', 'pit_out_time',
            'track_status', 'track_status_code',
            'is_yellow_flag', 'is_safety_car', 'is_vsc', 'is_red_flag',
            'air_temp', 'track_temp', 'humidity', 'rainfall', 'is_wet_race', 'is_accurate',
        ]
        final_cols = [c for c in final_cols if c in laps.columns]
        return laps[final_cols]

    except Exception as e:
        print(f'  PARSE ERROR {year} R{round_number}: {e}')
        import traceback
        traceback.print_exc()
        return None

print('Extraction function ready.')

Extraction function ready.


## 3. Collect All Races

In [47]:
# Load progress
if PROGRESS_FILE.exists():
    progress_df = pd.read_csv(PROGRESS_FILE)
    completed = set(zip(progress_df['season'], progress_df['round']))
    print(f'Resuming — {len(completed)} races done')
else:
    completed = set()
    print('Starting fresh')

# Load existing output
if OUTPUT_FILE.exists() and len(completed) > 0:
    all_data = [pd.read_csv(OUTPUT_FILE)]
    print(f'Loaded: {all_data[0].shape[0]:,} rows')
else:
    all_data = []

races_to_collect = schedule_df[
    ~schedule_df.apply(lambda r: (r['season'], r['round']) in completed, axis=1)
]
print(f'Remaining: {len(races_to_collect)} races\n')

progress_records = []

for _, row in tqdm(races_to_collect.iterrows(), total=len(races_to_collect), desc='Races'):
    year = int(row['season'])
    rnd = int(row['round'])
    event_name = row['event_name']
    country = row['country']
    location = row['location']
    date = row['date']

    print(f'  {year} R{rnd:02d} {event_name}', end=' ... ')

    race_df = extract_race_data(year, rnd, event_name, country, location, date)

    if race_df is not None and not race_df.empty:
        all_data.append(race_df)
        progress_records.append({'season': year, 'round': rnd, 'event_name': event_name})

        combined = pd.concat(all_data, ignore_index=True)
        combined.to_csv(OUTPUT_FILE, index=False)

        all_progress = list(completed) + [(r['season'], r['round']) for r in progress_records]
        pd.DataFrame(
            [{'season': s, 'round': r} for s, r in all_progress]
        ).to_csv(PROGRESS_FILE, index=False)

        print(f'✓ ({len(race_df):,} rows)')
    else:
        print('SKIP')

    time.sleep(DELAY_BETWEEN_SESSIONS)

print(f'\n✅ Done')

Resuming — 80 races done
Loaded: 87,818 rows
Remaining: 12 races



Races:   0%|          | 0/12 [00:00<?, ?it/s]

core           INFO 	Loading data for Belgian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Belgian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info


  2025 R13 Belgian Grand Prix ... 

req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core           INFO 	Processing timing data...
INFO:fastf1.fastf1.core:Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '81'
core        WARNING 	Fixed incorrect tyre stint information for driver '4'
core        WARNING 	Fixed incorrect tyre stint information for driver '16'
core        WARNING 	Fixed incorrect tyre stint information for 

...

✓ (1,067 rows)


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


  2025 R24 Abu Dhabi Grand Prix ... 

req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F

✓ (1,156 rows)

✅ Done


## 4. Validate

In [48]:
df = pd.read_csv(OUTPUT_FILE)

print('Shape:', df.shape)
print('\nRaces per season:', df.groupby('season')['round'].nunique().to_dict())
print('\nColumns:', df.columns.tolist())
print('\nDNF rate:', df.drop_duplicates(subset=['season', 'round', 'driver'])['dnf'].mean())
df.head(2)

Shape: (101290, 45)

Races per season: {2022: 22, 2023: 22, 2024: 24, 2025: 24}

Columns: ['season', 'round', 'event_name', 'country', 'location', 'date', 'is_street_circuit', 'total_race_laps', 'driver', 'driver_number', 'driver_abbrev', 'team', 'grid_position', 'final_position', 'points_scored', 'laps_completed', 'status', 'dnf', 'dnf_reason', 'dnf_lap', 'lap_number', 'position', 'lap_time_seconds', 'sector1_seconds', 'sector2_seconds', 'sector3_seconds', 'tire_compound', 'tire_age', 'is_fresh_tire', 'stint', 'is_pit_lap', 'pit_in_time', 'pit_out_time', 'track_status', 'track_status_code', 'is_yellow_flag', 'is_safety_car', 'is_vsc', 'is_red_flag', 'air_temp', 'track_temp', 'humidity', 'rainfall', 'is_wet_race', 'is_accurate']

DNF rate: 0.29518072289156627


,season,round,event_name,country,location,date,is_street_circuit,total_race_laps,driver,driver_number,...,is_yellow_flag,is_safety_car,is_vsc,is_red_flag,air_temp,track_temp,humidity,rainfall,is_wet_race,is_accurate
0,2022,1,Bahrain Grand Prix,Bahrain,Sakhir,2022-03-20,False,57.0,VER,1,...,False,False,False,False,23.9,29.0,26.0,False,False,False
1,2022,1,Bahrain Grand Prix,Bahrain,Sakhir,2022-03-20,False,57.0,VER,1,...,False,False,False,False,23.8,29.0,27.0,False,False,True


In [49]:
import pandas as pd

df = pd.read_csv('../data/raw/raw_f1_data_2022_2025.csv')

print("=== OVERALL ===")
print(f"Total rows: {len(df):,}")
print(f"Seasons: {sorted(df['season'].unique())}")

print("\n=== ROWS PER SEASON ===")
print(df.groupby('season').size())

print("\n=== RACES PER SEASON ===")
print(df.groupby('season')['round'].nunique())

print("\n=== NULLS IN CRITICAL COLUMNS ===")
critical = ['final_position', 'grid_position', 'lap_time_seconds', 'tire_compound', 'air_temp']
print(df[critical].isnull().sum())

print("\n=== FINAL POSITION RANGE PER SEASON ===")
race_level = df.drop_duplicates(subset=['season', 'round', 'driver'])
print(race_level.groupby('season')['final_position'].agg(['min', 'max', 'count']))

print("\n=== DNF RATE PER SEASON ===")
print(race_level.groupby('season')['dnf'].mean().round(3))

print("\n=== TIRE COMPOUNDS ===")
print(df['tire_compound'].value_counts())

print("\n=== AIR TEMP SAMPLE (first non-null) ===")
print(df['air_temp'].describe())

=== OVERALL ===
Total rows: 101,290
Seasons: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

=== ROWS PER SEASON ===
season
2022    23577
2023    24420
2024    26604
2025    26689
dtype: int64

=== RACES PER SEASON ===
season
2022    22
2023    22
2024    24
2025    24
Name: round, dtype: int64

=== NULLS IN CRITICAL COLUMNS ===
final_position         0
grid_position          0
lap_time_seconds    1539
tire_compound        780
air_temp               0
dtype: int64

=== FINAL POSITION RANGE PER SEASON ===
        min   max  count
season                  
2022    1.0  20.0    438
2023    1.0  20.0    436
2024    1.0  20.0    476
2025    1.0  20.0    476

=== DNF RATE PER SEASON ===
season
2022    0.164
2023    0.303
2024    0.397
2025    0.307
Name: dnf, dtype: float64

=== TIRE COMPOUNDS ===
tire_compound
HARD            44364
MEDIUM          36980
SOFT            12760
INTERMEDIATE     5945
WET               461
Name: count, dtype: int64

=== AIR TEMP SAMPLE (first no

In [50]:
race_level = df.drop_duplicates(subset=['season', 'round', 'driver'])

print("=== STATUS VALUE COUNTS ===")
print(race_level['status'].value_counts().head(20))

print("\n=== DNF REASON VALUE COUNTS ===")
print(race_level['dnf_reason'].value_counts().head(20))

print("\n=== DNF RATE PER SEASON ===")
print(race_level.groupby('season')['dnf'].mean().round(3))

print("\n=== SAMPLE DNF=TRUE ROWS ===")
print(race_level[race_level['dnf']==True][['season','round','event_name','driver','status','dnf_reason']].head(20))

=== STATUS VALUE COUNTS ===
status
Finished            1200
Lapped               297
Retired              153
+1 Lap                80
Collision damage      14
Accident              11
Collision             10
Disqualified          10
Engine                 7
+2 Laps                6
Power Unit             5
Hydraulics             4
Gearbox                4
Undertray              3
Water leak             2
Spun off               2
Water pressure         2
Fuel pressure          2
Oil leak               1
Differential           1
Name: count, dtype: int64

=== DNF REASON VALUE COUNTS ===
dnf_reason
Lapped              297
Retired             153
Collision damage     14
Accident             11
Collision            10
Disqualified         10
Engine                7
Power Unit            5
Gearbox               4
Hydraulics            4
Undertray             3
Spun off              2
Water leak            2
Water pressure        2
Fuel pressure         2
Oil leak              1
Differentia